# SQ1 calibration — DBB threshold tuning

Run after Ahmed fills `data/sq1_manual_labels.csv`. This notebook:

1. Joins manual labels to per-scene DBB means.
2. Searches for the DBB threshold that maximizes agreement between the DBB-derived
   binary flag (`dust` vs `not dust`) and manual labels (collapsing `heavy_dust` and
   `light_haze` → dust; `clean` → not dust; `cloud` and `mixed` excluded).
3. Reports agreement rate and confusion matrix.
4. Saves the chosen threshold to `data/sq1_threshold.json` for downstream sessions.

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA = Path('../data')
labels = pd.read_csv(DATA / 'sq1_manual_labels.csv')
values = pd.read_csv(DATA / 'sq1_dbb_values.csv')
df = labels.merge(values[['scene_id', 'dbb_mean']], on='scene_id')
df.head()

In [ ]:
# Collapse to binary, drop ambiguous
DUST_LABELS = {'heavy_dust', 'light_haze'}
CLEAN_LABELS = {'clean'}
df['truth'] = df['label'].map(lambda x: 1 if x in DUST_LABELS else (0 if x in CLEAN_LABELS else np.nan))
binary = df.dropna(subset=['truth']).copy()
print(f'Usable scenes: {len(binary)} ({(binary.truth==1).sum()} dust, {(binary.truth==0).sum()} clean)')

In [ ]:
# Threshold sweep
thresholds = np.linspace(binary.dbb_mean.min(), binary.dbb_mean.max(), 200)
agreements = []
for t in thresholds:
    pred = (binary.dbb_mean > t).astype(int)
    agreements.append((pred == binary.truth).mean())
agreements = np.array(agreements)
best_idx = int(agreements.argmax())
best_t = float(thresholds[best_idx])
best_agreement = float(agreements[best_idx])
print(f'Best threshold: {best_t:+.4f}   agreement: {best_agreement:.1%}')

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(thresholds, agreements)
ax.axvline(best_t, color='r', ls='--', label=f't={best_t:+.3f}')
ax.set_xlabel('DBB threshold')
ax.set_ylabel('agreement rate')
ax.legend()
fig.tight_layout()
fig.savefig('../figures/sq1_threshold_sweep.png', dpi=120)

In [ ]:
# Save
(DATA / 'sq1_threshold.json').write_text(json.dumps({
    'threshold': best_t,
    'agreement': best_agreement,
    'n_scenes': int(len(binary)),
    'rule': 'dbb_mean > threshold => flagged dust',
}, indent=2))